In [1]:
#run to start
import numpy as np
import ROOT
import time
import datetime
import calendar

ROOT.gStyle.SetOptStat(0)

def seconds_in_year(year):
    year = int(year)
    if year % 4 == 0:
        return 366 * 24 * 60 * 60
    else:
        return 365 * 24 * 60 * 60


UoB_year_files = {
    "2015": ["UoB_2015_1.root","UoB_2015_2.root","UoB_2015_3.root","UoB_2015_4.root","UoB_2015_5.root"],
    "2016": ["UoB_2016_1.root","UoB_2016_2.root","UoB_2016_3.root"],
    "2017": ["UoB_2017_1.root","UoB_2017_2.root","UoB_2017_3.root","UoB_2017_4.root"],
    "2018": ["UoB_2018_1.root","UoB_2018_2.root","UoB_2018_3.root"],
    "2019": ["UoB_2019_1.root","UoB_2019_2.root","UoB_2019_3.root","UoB_2019_4.root"],
    "2020": ["UoB_2020_1.root"],
    "2022": ["UoB_2022_1.root","UoB_2022_2.root","UoB_2022_3.root","UoB_2022_4.root"],
    "2023": ["UoB_2023_1.root"],
    "2024": ["UoB_2024_1.root","UoB_2024_2.root","UoB_2024_3.root"],
    "2025": ["UoB_2025_1.root","UoB_2025_2.root","UoB_2025_3.root","UoB_2025_4.root","UoB_2025_5.root"]
}

Bromsgrove_year_files = {
    "2020": ["Bromsgrove_2020_1.root","Bromsgrove_2020_2.root","Bromsgrove_2020_3.root"],
    "2021": ["Bromsgrove_2021_1.root","Bromsgrove_2021_2.root","Bromsgrove_2021_3.root","Bromsgrove_2021_4.root","Bromsgrove_2021_5.root",],
    "2022": ["Bromsgrove_2022_1.root",],
    "2023": ["Bromsgrove_2023_1.root","Bromsgrove_2023_2.root","Bromsgrove_2023_3.root"],
    "2024": ["Bromsgrove_2024_1.root","Bromsgrove_2024_2.root","Bromsgrove_2024_3.root"],
    "2025": ["Bromsgrove_2025_1.root","Bromsgrove_2025_2.root"]
}

IOP_year_files = {
    "2019": ["IOP_2019_1.root","IOP_2019_2.root","IOP_2019_3.root","IOP_2019_4.root"],
    "2020": ["IOP_2020_1.root","IOP_2020_2.root","IOP_2020_3.root","IOP_2020_4.root","IOP_2020_5.root"],
    "2021": ["IOP_2021_1.root","IOP_2021_2.root","IOP_2021_3.root","IOP_2021_4.root"],
    "2022": ["IOP_2022_1.root","IOP_2022_2.root","IOP_2022_3.root"],
    "2023": ["IOP_2023_1.root","IOP_2023_2.root","IOP_2023_3.root","IOP_2023_4.root","IOP_2023_5.root","IOP_2023_6.root"],
    "2024": ["IOP_2024_1.root","IOP_2024_2.root","IOP_2024_3.root","IOP_2024_4.root","IOP_2024_5.root"]
}



def year_length(year_files):
    year_lengths = {}
    total_span = 0
    for year in year_files:
        year_lengths[year] = seconds_in_year(year)
        total_span += year_lengths[year]
    return year_lengths, total_span


UoB_year_lengths, UoB_total_span = year_length(UoB_year_files)
Bromsgrove_year_lengths, Bromsgrove_total_span = year_length(Bromsgrove_year_files)
IOP_year_lengths, IOP_total_span = year_length(IOP_year_files)

UNIX_year_start = {}

for year in range(2015, 2026):
    jan1 = datetime.datetime(year, 1, 1, 0, 0, 0)
    UNIX_year_start[str(year)] = calendar.timegm(jan1.timetuple())

UNIX_year_start


def year_hist(year, files, UNIX_year_start, colour, bins = 180):
    chain = ROOT.TChain("events")

    total_time = 0
    jan1_UNIX = UNIX_year_start[year]
    for file in files:
        chain.Add(file)
        
        f = ROOT.TFile(file)
        t = f.Get("events")
        
        tmin = t.GetMinimum("UNIX")
        tmax = t.GetMaximum("UNIX")

        total_time += (tmax - tmin)

        f.Close()
    year_seconds = seconds_in_year(year)

    fraction = total_time/year_seconds

    hist = ROOT.TH1D(f"h_{year}", f"{year};Seconds since Jan 1;Scaled counts", bins, 0, year_seconds)
    
    chain.Draw(f"(UNIX - {jan1_UNIX}) >> h_{year}", "", "goff")
    if hist.Integral() != 0:
        hist.Scale(fraction / hist.Integral())
    hist.SetLineWidth(2)
    hist.SetLineColor(colour)

    return hist


def year_hist_alongside(year, files, UNIX_year_start, colour, offset, total_span, bins=180):

    chain = ROOT.TChain("events")
    total_time = 0.0

    for file in files:
        chain.Add(file)

        f = ROOT.TFile(file)
        t = f.Get("events")

        tmin = t.GetMinimum("UNIX")
        tmax = t.GetMaximum("UNIX")

        total_time += (tmax - tmin)
        f.Close()

    year_seconds = seconds_in_year(year)
    fraction = total_time / year_seconds


    hist = ROOT.TH1D(f"h_{year}", f";Time (years placed side-by-side);Scaled counts", bins, 0, total_span)

    jan1_UNIX = UNIX_year_start[year]
    chain.Draw(f"(UNIX - {jan1_UNIX} + {offset}) >> h_{year}", "", "goff")

    if hist.Integral() != 0:
        hist.Scale(fraction / hist.Integral())

    hist.SetLineWidth(2)
    hist.SetLineColor(colour)
    
    return hist




def sin_fit(x, ps):
    return ps[0]*ROOT.TMath.Sin(ps[1]*x[0]+ps[2])+ps[3]



def hist_sin_fit(histograms, total_span, bins, omega = False):
    h_global = ROOT.TH1D("h_global_temp", "", bins, 0, total_span)
    h_global.SetDirectory(0)

    for _, h in histograms:
        h_global.Add(h)

    xmin = h_global.GetXaxis().GetXmin()
    xmax = h_global.GetXaxis().GetXmax()

    fit = ROOT.TF1("global_sin_temp", sin_fit, xmin, xmax, 4)
    fit.SetNpx(5000)

    seconds_per_year = 365.25 * 24 * 3600

    offset_guess = h_global.Integral() / h_global.GetNbinsX()
    amplitude_guess = 0.1 * offset_guess
    omega_guess = 2*np.pi/seconds_per_year

    fit.SetParameters(amplitude_guess, omega_guess, 0, offset_guess)

    if omega:
        fit.FixParameter(1, omega_guess)
    else:
        pass

    h_global.Fit(fit, "W0")

    fit.SetLineColor(ROOT.kBlack)
    fit.SetLineWidth(2)
    fit.Draw("SAME")

    return fit



colours = [
    ROOT.kBlue,
    ROOT.kRed,
    ROOT.kGreen+2,
    ROOT.kMagenta,
    ROOT.kOrange+7,
    ROOT.kYellow-6,
    ROOT.kViolet,
    ROOT.kTeal+3,
    ROOT.kPink+7,
    ROOT.kAzure+1
]



def extract_lightning(data_file):
    file = np.loadtxt(data_file, skiprows=1, usecols=range(4), delimiter = ',')
    lightning = file[:,3]
    monthly_lightning = []
    for i in range(12):
        monthly_lightning.append(int(np.sum(lightning[i::12])))
    return np.array(monthly_lightning)



Bham_lightning_files = {
    "2015": "Data/2015_bham_subset.csv",
    "2016": "Data/2016_bham_subset.csv",
    "2017": "Data/2017_bham_subset.csv",
    "2018": "Data/2018_bham_subset.csv",
    "2019": "Data/2019_bham_subset.csv",
    "2020": "Data/2020_bham_subset.csv",
    "2021": "Data/2021_bham_subset.csv",
    "2022": "Data/2022_bham_subset.csv",
    "2023": "Data/2023_bham_subset.csv",
    "2024": "Data/2024_bham_subset.csv",
    "2025": "Data/2025_bham_subset.csv"
}



In [2]:
#def extract_data(file):

def extract_data(file):
    data = np.loadtxt(file, skiprows=27, usecols=range(2, 23))

    UNIX = data[:,0]
    nanoseconds = data[:,1]

    pulseheights = data[:,2:6]
    integral = data[:,6:10]
    mips = data[:,10:14]
    arrivalTimes = data[:,14:18]

    triggerTimes = data[:,18]
    zenith = data[:,19]
    azimuth = data[:,20]
    return UNIX, nanoseconds, pulseheights, integral, mips, arrivalTimes, triggerTimes, zenith, azimuth

UNIX, nanoseconds, pulseheights, integral, mips, arrivalTimes, triggerTimes, zenith, azimuth = extract_data('Events21Jan25.tsv')

In [22]:
#def text_to_root(txtfile, rootfile):

def text_to_root(txtfile, rootfile):
    data = np.loadtxt(txtfile, skiprows=27, usecols=range(2, 23))

    f = ROOT.TFile(rootfile, "RECREATE")
    t = ROOT.TTree("events", "Converted data")

    UNIX = np.zeros(1, dtype=np.float64)
    nanoseconds = np.zeros(1, dtype=np.float64)

    pulseheights = np.zeros(4, dtype=np.float64)
    integral = np.zeros(4, dtype=np.float64)
    mips = np.zeros(4, dtype=np.float64)
    arrivalTimes = np.zeros(4, dtype=np.float64)

    triggerTimes = np.zeros(1, dtype=np.float64)
    zenith = np.zeros(1, dtype=np.float64)
    azimuth = np.zeros(1, dtype=np.float64)
    
    t.Branch("UNIX", UNIX, "UNIX/D")
    t.Branch("nanoseconds", nanoseconds, "nanoseconds/D")
    
    t.Branch("pulseheights", pulseheights, "pulseheights[4]/D")
    t.Branch("integral", integral, "integral[4]/D")
    t.Branch("mips", mips, "mips[4]/D")
    t.Branch("arrivalTimes", arrivalTimes, "arrivalTimes[4]/D")
    
    t.Branch("triggerTimes", triggerTimes, "triggerTimes/D")
    t.Branch("zenith", zenith, "zenith/D")
    t.Branch("azimuth", azimuth, "azimuth/D")

    for row in data:
        UNIX[0] = row[0]
        nanoseconds[0] = row[1]

        pulseheights[:] = row[2:6]
        integral[:] = row[6:10]
        mips[:] = row[10:14]
        arrivalTimes[:] = row[14:18]

        triggerTimes[0] = row[18]
        zenith[0] = row[19]
        azimuth[0] = row[20]

        t.Fill()

    t.Write()
    f.Close()


In [23]:
#def Lightning_to_root(txtfile, rootfile):

def Lightning_to_root(txtfile, rootfile):
    data = np.loadtxt(txtfile, skiprows=1, usecols=range(4), delimiter = ',')

    f = ROOT.TFile(rootfile, "RECREATE")
    t = ROOT.TTree("Lightning", "Converted data")

    latitude = np.zeros(1, dtype=np.float64)
    longitude = np.zeros(1, dtype=np.float64)
    month = np.zeros(1, dtype=np.float64)
    thunder_hours = np.zeros(1, dtype=np.float64)
    
    t.Branch("latitude", latitude, "latitude/D")
    t.Branch("longitude", longitude, "longitude/D")
    t.Branch("month", month, "month/D")
    t.Branch("thunder_hours", thunder_hours, "thunder_hours/D")

    for row in data:
        latitude[0] = row[0]
        longitude[0] = row[1]
        month[0] = row[2]
        thunder_hours[0] = row[3]

        t.Fill()

    t.Write()
    f.Close()

file = np.loadtxt("Data/bham.csv", skiprows=1, usecols=range(4), delimiter = ',')
file

array([[51.975,  1.025,  1.   ,  0.   ],
       [51.975,  1.025,  2.   ,  0.   ],
       [51.975,  1.025,  3.   ,  0.   ],
       ...,
       [51.025,  2.475, 10.   ,  2.   ],
       [51.025,  2.475, 11.   ,  0.   ],
       [51.025,  2.475, 12.   ,  0.   ]], shape=(7200, 4))

In [ ]:
#Lightning_to_root("Data/bham.csv", "BhamLightning.root")

In [29]:
#text_to_root("UOB_2025_5.tsv", "UOB_2025_5.root")
start_time = time.perf_counter()

#text_to_root("UOB_2017_4.tsv", "UOB_2017_4.root")

stop_time =  time.perf_counter()
print(f"Code took {1e3*(stop_time-start_time):.5} ms to run")

#Audio(audio_data, rate=framerate, autoplay=True)

Code took 0.060542 ms to run


In [27]:
#histogram UoB_2018_1

f = ROOT.TFile("UoB_2018_1.root")
t = f.Get("events")

xmin = t.GetMinimum("UNIX")
xmax = t.GetMaximum("UNIX")

h = ROOT.TH1D("h", "Events;UNIX time;Counts", 60, xmin, xmax)

t.Draw("UNIX >> h")

c = ROOT.TCanvas()

h.SetFillColor(ROOT.TColor.GetColor("#d399f2"))
h.SetLineColor(ROOT.TColor.GetColor("#ad73f0"))
h.SetLineWidth(2)

h.SetMinimum(0)
h.Draw()

c.Draw()
print((xmax-xmin)/3600/24)

164.9999537037037


In [8]:
from IPython.lib.display import Audio

framerate = 4410
play_time_seconds = 1

t = np.linspace(0, play_time_seconds, framerate*play_time_seconds)
audio_data = np.sin(2*np.pi*300*t) + np.sin(2*np.pi*240*t)
Audio(audio_data, rate=framerate, autoplay=True)

In [21]:
def seconds_in_year(year):
    year = int(year)
    if year % 4 == 0:
        return 366 * 24 * 60 * 60
    else:
        return 365 * 24 * 60 * 60

In [29]:
def year_length(year_files,total_span = 0,year_lengths = {}):
    for year in year_files:
        year_lengths[year] = seconds_in_year(year)
        total_span += year_lengths[year]
    return year_lengths, total_span

In [2]:
#def year_hist on top

def year_hist(year, files, UNIX_year_start, colour, bins = 180):
    chain = ROOT.TChain("events")

    total_time = 0
    jan1_UNIX = UNIX_year_start[year]
    for file in files:
        chain.Add(file)
        
        f = ROOT.TFile(file)
        t = f.Get("events")
        
        tmin = t.GetMinimum("UNIX")
        tmax = t.GetMaximum("UNIX")

        total_time += (tmax - tmin)

        f.Close()
    
    year_seconds = seconds_in_year(year)

    fraction = total_time/year_seconds

    hist = ROOT.TH1D(f"h_{year}", f"{year};Seconds since Jan 1;Scaled counts", bins, 0, year_seconds)
    
    chain.Draw(f"(UNIX - {jan1_UNIX}) >> h_{year}", "", "goff")
    if hist.Integral() != 0:
        hist.Scale(fraction / hist.Integral())
    hist.SetLineWidth(2)
    hist.SetLineColor(colour)

    return hist


In [12]:
#def year_hist_alongside

def year_hist_alongside(year, files, UNIX_year_start, colour, offset, total_span, bins=180):

    chain = ROOT.TChain("events")
    total_time = 0.0

    for file in files:
        chain.Add(file)

        f = ROOT.TFile(file)
        t = f.Get("events")

        tmin = t.GetMinimum("UNIX")
        tmax = t.GetMaximum("UNIX")

        total_time += (tmax - tmin)
        f.Close()

    year_seconds = seconds_in_year(year)
    fraction = total_time / year_seconds


    hist = ROOT.TH1D(f"h_{year}", f";Time (years placed side-by-side);Scaled counts", bins, 0, total_span)

    jan1_UNIX = UNIX_year_start[year]
    chain.Draw(f"(UNIX - {jan1_UNIX} + {offset}) >> h_{year}", "", "goff")

    if hist.Integral() != 0:
        hist.Scale(fraction / hist.Integral())

    hist.SetLineWidth(2)
    hist.SetLineColor(colour)
    
    return hist


In [45]:
def hist_sin_fit(histograms, total_span, bins, omega = False):
    h_global = ROOT.TH1D("h_global_temp", "", bins, 0, total_span)
    h_global.SetDirectory(0)

    for _, h in histograms:
        h_global.Add(h)

    xmin = h_global.GetXaxis().GetXmin()
    xmax = h_global.GetXaxis().GetXmax()

    fit = ROOT.TF1("global_sin_temp", sin_fit, xmin, xmax, 4)
    fit.SetNpx(5000)

    seconds_per_year = 365.25 * 24 * 3600

    offset_guess = h_global.Integral() / h_global.GetNbinsX()
    amplitude_guess = 0.5 * offset_guess
    omega_guess = 2*np.pi/seconds_per_year

    fit.SetParameters(amplitude_guess, omega_guess, 0, offset_guess)

    if omega:
        fit.FixParameter(1, omega_guess)
    else:
        pass

    h_global.Fit(fit, "W0")

    fit.SetLineColor(ROOT.kBlack)
    fit.SetLineWidth(2)
    fit.Draw("SAME")

    return fit

In [5]:
#UNIX Dictionary

UNIX_year_start = {}

for year in range(2015, 2026):
    jan1 = datetime.datetime(year, 1, 1, 0, 0, 0)
    UNIX_year_start[str(year)] = calendar.timegm(jan1.timetuple())

UNIX_year_start

{'2015': 1420070400,
 '2016': 1451606400,
 '2017': 1483228800,
 '2018': 1514764800,
 '2019': 1546300800,
 '2020': 1577836800,
 '2021': 1609459200,
 '2022': 1640995200,
 '2023': 1672531200,
 '2024': 1704067200,
 '2025': 1735689600}

In [1]:
#collected files
UoB_year_files = {
    "2015": ["UoB_2015_1.root","UoB_2015_2.root","UoB_2015_3.root","UoB_2015_4.root","UoB_2015_5.root"],
    "2016": ["UoB_2016_1.root","UoB_2016_2.root","UoB_2016_3.root"],
    "2017": ["UoB_2017_1.root","UoB_2017_2.root","UoB_2017_3.root","UoB_2017_4.root"],
    "2018": ["UoB_2018_1.root","UoB_2018_2.root","UoB_2018_3.root"],
    "2019": ["UoB_2019_1.root","UoB_2019_2.root","UoB_2019_3.root","UoB_2019_4.root"],
    "2020": ["UoB_2020_1.root"],
    "2022": ["UoB_2022_1.root","UoB_2022_2.root","UoB_2022_3.root","UoB_2022_4.root"],
    "2023": ["UoB_2023_1.root"],
    "2024": ["UoB_2024_1.root","UoB_2024_2.root","UoB_2024_3.root"],
    "2025": ["UoB_2025_1.root","UoB_2025_2.root","UoB_2025_3.root","UoB_2025_4.root","UoB_2025_5.root"]
}

Bromsgrove_year_files = {
    "2020": ["Bromsgrove_2020_1.root","Bromsgrove_2020_2.root","Bromsgrove_2020_3.root"],
    "2021": ["Bromsgrove_2021_1.root","Bromsgrove_2021_2.root","Bromsgrove_2021_3.root","Bromsgrove_2021_4.root","Bromsgrove_2021_5.root",],
    "2022": ["Bromsgrove_2022_1.root",],
    "2023": ["Bromsgrove_2023_1.root","Bromsgrove_2023_2.root","Bromsgrove_2023_3.root"],
    "2024": ["Bromsgrove_2024_1.root","Bromsgrove_2024_2.root","Bromsgrove_2024_3.root"],
    "2025": ["Bromsgrove_2025_1.root","Bromsgrove_2025_2.root"]
}

IOP_year_files = {
    "2019": ["IOP_2019_1.root","IOP_2019_2.root","IOP_2019_3.root","IOP_2019_4.root"],
    "2020": ["IOP_2020_1.root","IOP_2020_2.root","IOP_2020_3.root","IOP_2020_4.root","IOP_2020_5.root"],
    "2021": ["IOP_2021_1.root","IOP_2021_2.root","IOP_2021_3.root","IOP_2021_4.root"],
    "2022": ["IOP_2022_1.root","IOP_2022_2.root","IOP_2022_3.root"],
    "2023": ["IOP_2023_1.root","IOP_2023_2.root","IOP_2023_3.root","IOP_2023_4.root","IOP_2023_5.root","IOP_2023_6.root"],
    "2024": ["IOP_2024_1.root","IOP_2024_2.root","IOP_2024_3.root","IOP_2024_4.root","IOP_2024_5.root"]
}

In [7]:
colours = [
    ROOT.kBlue,
    ROOT.kRed,
    ROOT.kGreen+2,
    ROOT.kMagenta,
    ROOT.kOrange+7,
    ROOT.kYellow-6,
    ROOT.kViolet,
    ROOT.kTeal+3,
    ROOT.kPink+7,
    ROOT.kAzure+1
]

In [179]:
#2018 v 2023 scaled wrong
bins = 365
year_seconds = 365 * 24 * 60 * 60

t2018 = ROOT.TChain("events")
t2018.Add("UoB_2018_1.root")
t2018.Add("UoB_2018_2.root")
t2018.Add("UoB_2018_3.root")

f2023 = ROOT.TFile("UoB_2023_1.root")
t2023 = f2023.Get("events")

min2018 = t2018.GetMinimum("UNIX")
min2023 = t2023.GetMinimum("UNIX")


h2018 = ROOT.TH1D("h2018", "2018 vs 2023;Seconds since start of year;Scaled Counts", bins, 0, year_seconds)

h2023 = ROOT.TH1D("h2023", "2018 vs 2023;Seconds since start of year;Scaled Counts", bins, 0, year_seconds)

t2018.Draw(f"(UNIX - {min2018}) >> h2018", "", "goff")
t2023.Draw(f"(UNIX - {min2023}) >> h2023", "", "goff")

if h2018.Integral() != 0:
    h2018.Scale(1.0 / h2018.Integral())

if h2023.Integral() != 0:
    h2023.Scale(1.0 / h2023.Integral())

h2018.SetLineColor(ROOT.kBlue)
h2018.SetLineWidth(2)

h2023.SetLineColor(ROOT.kRed)
h2023.SetLineWidth(2)

ymax = 1.1 * max(h2018.GetMaximum(), h2023.GetMaximum())
h2018.SetMaximum(ymax)
h2018.SetMinimum(0)

c1 = ROOT.TCanvas()
h2018.Draw("HIST")
h2023.Draw("HIST SAME")

leg = ROOT.TLegend(0.7, 0.75, 0.9, 0.9)
leg.AddEntry(h2018, "2018", "l")
leg.AddEntry(h2023, "2023", "l")
leg.Draw()

c1.Draw()


In [2]:
#using year_hist

h2023 = year_hist("2023", UoB_year_files["2023"], UNIX_year_start, ROOT.kRed)

h2017 = year_hist("2017", UoB_year_files["2017"], UNIX_year_start, ROOT.kGreen+2)
#h2020 = year_hist("2020", UoB_year_files["2020"], UNIX_year_start, ROOT.kAzure+2)

ymax = 1.1 * max(h2017.GetMaximum(), h2023.GetMaximum())
h2023.SetMaximum(ymax)
h2023.SetMinimum(0)

c = ROOT.TCanvas("c", "", 800, 600)

h2023.Draw("HIST")
#h2020.Draw("HIST SAME")
h2017.Draw("HIST SAME")

leg = ROOT.TLegend(0.75, 0.75, 0.9, 0.9)
leg.AddEntry(h2017, "2017", "l")
#leg.AddEntry(h2020, "2020", "l")
leg.AddEntry(h2023, "2023", "l")
leg.Draw()

c.Draw()


In [7]:
#UoB using year_hist looped

histograms = []
bins = 180

for i,year in enumerate(UoB_year_files):
    colour = colours[i]
    h = year_hist(year, UoB_year_files[year], UNIX_year_start, colour, bins)
    histograms.append((year, h))

ymax = 1.2 * max(h.GetMaximum() for year, h in histograms)
c_all = ROOT.TCanvas("c_all_years", "All Years", 900, 600)

first = True
for year, h in histograms:
    h.SetMaximum(ymax)
    h.SetMinimum(0)

    if first:
        h.Draw("P")
        first = False
    else:
        h.Draw("P SAME")

leg = ROOT.TLegend(0.1, 0.82, 0.9, 0.9)
leg.SetNColumns(len(histograms))

for year, h in histograms:
    leg.AddEntry(h, year, "l")

leg.Draw()

c_all.Draw()

Warning in <TCanvas::Constructor>: Deleting canvas with same name: c_all_years


In [6]:
#Bromsgrove using year_hist looped 

histograms = []

for i,year in enumerate(Bromsgrove_year_files):
    colour = colours[i]
    h = year_hist(year, Bromsgrove_year_files[year], UNIX_year_start, colour)
    histograms.append((year, h))

ymax = 1.2 * max(h.GetMaximum() for year, h in histograms)
c_all = ROOT.TCanvas("c_all_years", "All Years", 900, 600)

first = True
for year, h in histograms:
    h.SetMaximum(ymax)
    h.SetMinimum(0)

    if first:
        h.Draw("P")
        first = False
    else:
        h.Draw("P SAME")

leg = ROOT.TLegend(0.1, 0.82, 0.9, 0.9)
leg.SetNColumns(len(histograms))

for year, h in histograms:
    leg.AddEntry(h, year, "l")

leg.Draw()

c_all.Draw()

Warning in <TROOT::Append>: Replacing existing TH1: h_2025 (Potential memory leak).


In [27]:
#IOP using year_hist looped

histograms = []

for i,year in enumerate(IOP_year_files):
    colour = colours[i]
    h = year_hist(year, IOP_year_files[year], UNIX_year_start, colour)
    histograms.append((year, h))

ymax = 1.2 * max(h.GetMaximum() for year, h in histograms)
c_all = ROOT.TCanvas("c_all_years", "All Years", 900, 600)

first = True
for year, h in histograms:
    h.SetMaximum(ymax)
    h.SetMinimum(0)

    if first:
        h.Draw("HIST")
        first = False
    else:
        h.Draw("HIST SAME")

leg = ROOT.TLegend(0.1, 0.82, 0.9, 0.9)
leg.SetNColumns(len(histograms))

for year, h in histograms:
    leg.AddEntry(h, year, "l")

leg.Draw()

c_all.Draw()

Warning in <TCanvas::Constructor>: Deleting canvas with same name: c_all_years


In [19]:
#UoB using year_hist_alongside

offset = 0
histograms = []

for i, year in enumerate(UoB_year_files):
    colour = colours[i]
    hist = year_hist_alongside(year, UoB_year_files[year], UNIX_year_start, colour, offset, UoB_total_span)
    histograms.append((year, hist))
    offset += UoB_year_lengths[year]

ymax = 1.2 * max(h.GetMaximum() for _, h in histograms)


c_side = ROOT.TCanvas("c_side_by_side", "Years Side By Side", 1100, 500)

first = True
for year, h in histograms:
    h.SetMaximum(ymax)
    h.SetMinimum(0)

    if first:
        h.Draw("HIST")
        first = False
    else:
        h.Draw("HIST SAME")


leg = ROOT.TLegend(0.1, 0.82, 0.9, 0.9)
leg.SetNColumns(len(histograms))

for year, h in histograms:
    leg.AddEntry(h, year, "l")

leg.Draw()

c_side.Draw()

Warning in <TROOT::Append>: Replacing existing TH1: h_2025 (Potential memory leak).
Warning in <TCanvas::Constructor>: Deleting canvas with same name: c_side_by_side


In [5]:
#UoB using year_hist_alongside with sin fit

offset = 0
histograms = []
bins = 500

for i, year in enumerate(UoB_year_files):
    colour = colours[i]
    hist = year_hist_alongside(year, UoB_year_files[year], UNIX_year_start, colour, offset, UoB_total_span, bins)
    histograms.append((year, hist))
    offset += UoB_year_lengths[year]

ymax = 1.2 * max(h.GetMaximum() for _, h in histograms)


c_side = ROOT.TCanvas("c_side_by_side", "Years Side By Side", 1100, 500)

first = True
for year, h in histograms:
    h.SetMaximum(ymax)
    h.SetMinimum(0)

    if first:
        h.Draw("HIST")
        first = False
    else:
        h.Draw("HIST SAME")


leg = ROOT.TLegend(0.1, 0.82, 0.9, 0.9)
leg.SetNColumns(len(histograms))

for year, h in histograms:
    leg.AddEntry(h, year, "l")

leg.Draw()

fix_omega = True
fit = hist_sin_fit(histograms, UoB_total_span, bins, fix_omega)

c_side.Draw()

****************************************
Minimizer is Minuit2 / Migrad
Chi2                      =   0.00632783
NDf                       =          393
Edm                       =  3.78686e-06
NCalls                    =           54
p0                        =   0.00180948   +/-   0.00029193  
p1                        =  1.99102e-07                      	 (fixed)
p2                        =     -1.36065   +/-   0.156882    
p3                        =    0.0187698   +/-   0.00020291  


Warning in <TROOT::Append>: Replacing existing TH1: h_2025 (Potential memory leak).
Warning in <TCanvas::Constructor>: Deleting canvas with same name: c_side_by_side


In [34]:
def draw_lightning_overlay(year, hist, offset, lightning_files):

    lightning = extract_lightning(lightning_files[year])

    year_seconds = seconds_in_year(year)
    month_seconds = year_seconds / 12

    lightning_max = max(lightning)
    hist_max = hist.GetMaximum()

    scale = 1.2 * hist_max / lightning_max  

    g_lightning = ROOT.TGraph(12)

    for i, val in enumerate(lightning):

        x = offset + (i + 0.5) * month_seconds
        y = val * scale

        g_lightning.SetPoint(i, x, y)

    g_lightning.SetMarkerStyle(20)
    g_lightning.SetMarkerSize(1.0)
    g_lightning.SetMarkerColor(ROOT.kRed)
    g_lightning.SetLineColor(ROOT.kRed)

    g_lightning.Draw("P SAME")   # points only


    xmax = hist.GetXaxis().GetXmax()

    axis = ROOT.TGaxis( xmax, 0, xmax, hist_max, 0, lightning_max, 510, "+L")

    axis.SetLineColor(ROOT.kRed)
    axis.SetLabelColor(ROOT.kRed)
    axis.SetTitle("Lightning hours per month")

    axis.Draw()


    return g_lightning, axis

In [40]:
def correlation(histograms, lightning_files):
    events_all = []
    lightning_all = []

    for year, hist in histograms:
        if year == "2025":
            continue

        events = [hist.GetBinContent(i+1) for i in range(12)]
        lightning = extract_lightning(lightning_files[year])

        events_all.extend(events)
        lightning_all.extend(lightning)

    corr = np.corrcoef(events_all, lightning_all)[0,1]
    return corr

In [41]:
#Bromsgrove using year_hist_alongside

offset = 0
histograms = []
bins = 48


for i, year in enumerate(Bromsgrove_year_files):
    if year == "2025":
        continue
    colour = colours[i]
    hist = year_hist_alongside(year, Bromsgrove_year_files[year], UNIX_year_start, colour, offset, Bromsgrove_total_span, bins)
    histograms.append((year, hist))
    offset += Bromsgrove_year_lengths[year]

ymax = 1.2 * max(h.GetMaximum() for _, h in histograms)


c_side = ROOT.TCanvas("c_side_by_side", "Years Side By Side", 1100, 500)

first = True
for year, h in histograms:
    h.SetMaximum(ymax)
    h.SetMinimum(0)

    if first:
        h.Draw("HIST")
        first = False
    else:
        h.Draw("HIST SAME")

offset = 0
lightning_graphs = []

for year, h in histograms:
    g, axis = draw_lightning_overlay(year, h, offset, Bham_lightning_files)

    lightning_graphs.append(g)

    offset += Bromsgrove_year_lengths[year]

leg = ROOT.TLegend(0.1, 0.82, 0.9, 0.9)
leg.SetNColumns(len(histograms))

for year, h in histograms:
    leg.AddEntry(h, year, "l")

#leg.AddEntry(lightning_graphs[0], "Lightning", "p")

leg.Draw()

c_side.Draw()
corr = correlation(histograms, Bham_lightning_files)
print(f"Correlation (events vs lightning) across all years: {corr:.3f}")

Correlation (events vs lightning) across all years: -0.105


Warning in <TROOT::Append>: Replacing existing TH1: h_2024 (Potential memory leak).
Warning in <TCanvas::Constructor>: Deleting canvas with same name: c_side_by_side


In [8]:
#Bromsgrove using year_hist_alongside with sin fit

offset = 0
histograms = []
bins = 200


for i, year in enumerate(Bromsgrove_year_files):
    colour = colours[i]
    hist = year_hist_alongside(year, Bromsgrove_year_files[year], UNIX_year_start, colour, offset, Bromsgrove_total_span, bins)
    histograms.append((year, hist))
    offset += Bromsgrove_year_lengths[year]

ymax = 1.2 * max(h.GetMaximum() for _, h in histograms)


c_side = ROOT.TCanvas("c_side_by_side", "Years Side By Side", 1100, 500)

first = True
for year, h in histograms:
    h.SetMaximum(ymax)
    h.SetMinimum(0)

    if first:
        h.Draw("HIST")
        first = False
    else:
        h.Draw("HIST SAME")


leg = ROOT.TLegend(0.1, 0.82, 0.9, 0.9)
leg.SetNColumns(len(histograms))

for year, h in histograms:
    leg.AddEntry(h, year, "l")

leg.Draw()

fix_omega = True
fit = hist_sin_fit(histograms, Bromsgrove_total_span, bins, fix_omega)

c_side.Draw()

****************************************
Minimizer is Minuit2 / Migrad
Chi2                      =   0.00240878
NDf                       =          177
Edm                       =  5.60191e-08
NCalls                    =           58
p0                        =   0.00302486   +/-   0.000392868 
p1                        =  1.99102e-07                      	 (fixed)
p2                        =     -1.78902   +/-   0.128651    
p3                        =    0.0289498   +/-   0.000276463 


Warning in <TROOT::Append>: Replacing existing TH1: h_2025 (Potential memory leak).
Warning in <TCanvas::Constructor>: Deleting canvas with same name: c_side_by_side


In [9]:
#IOP using year_hist_alongside

offset = 0
histograms = []
bins = 200

for i, year in enumerate(IOP_year_files):
    colour = colours[i]
    hist = year_hist_alongside(year, IOP_year_files[year], UNIX_year_start, colour, offset, IOP_total_span, bins)
    histograms.append((year, hist))
    offset += IOP_year_lengths[year]

ymax = 1.2 * max(h.GetMaximum() for _, h in histograms)


c_side = ROOT.TCanvas("c_side_by_side", "Years Side By Side", 1100, 500)

first = True
for year, h in histograms:
    h.SetMaximum(ymax)
    h.SetMinimum(0)

    if first:
        h.Draw("HIST")
        first = False
    else:
        h.Draw("HIST SAME")


leg = ROOT.TLegend(0.1, 0.82, 0.9, 0.9)
leg.SetNColumns(len(histograms))

for year, h in histograms:
    leg.AddEntry(h, year, "l")

leg.Draw()

c_side.Draw()

Warning in <TCanvas::Constructor>: Deleting canvas with same name: c_side_by_side


In [192]:
for i,year in enumerate(year_files):
    print(i,year)

0 2015
1 2016
2 2017
3 2018
4 2019
5 2020
6 2022
7 2023
8 2024
9 2025


In [2]:
import numpy as np
import ROOT
import time
import datetime
import calendar

In [1]:
import numpy as np
import time
import datetime
import calendar

In [2]:
import ROOT

In [12]:
#year_hist single year 2023 sin fit


f = ROOT.TFile(file)
t = f.Get("events")

f.Close()

h2023 = year_hist("2023", UoB_year_files["2023"], UNIX_year_start, ROOT.kYellow-6)

xmin = h2023.GetXaxis().GetXmin()
xmax = h2023.GetXaxis().GetXmax()

ymax = 1.1 * h2023.GetMaximum()
h2023.SetMaximum(ymax)
h2023.SetMinimum(0)

def sin_fit(x, ps):
    return ps[0]*ROOT.TMath.Sin(ps[1]*x[0]+ps[2])+ps[3]

fit = ROOT.TF1("fit2023", sin_fit, xmin, xmax, 4)

fit.SetParameters( 2e-3, 2*np.pi/(30e6), 0, 4e-3)
h2023.Fit(fit, "0")

c2023 = ROOT.TCanvas("c2023", "2023", 900, 600)

h2023.Draw("HIST")
fit.Draw("SAME")

c2023.Draw()

****************************************
Minimizer is Minuit2 / Migrad
Chi2                      =       149617
NDf                       =          176
Edm                       =  4.24132e-07
NCalls                    =          196
p0                        = -0.000656403   +/-   2.07296e-06 
p1                        =  1.92644e-07   +/-   6.80922e-10 
p2                        =      1.50222   +/-   0.0125575   
p3                        =   0.00548165   +/-   2.66367e-06 


Warning in <TROOT::Append>: Replacing existing TH1: h_2023 (Potential memory leak).


In [23]:
#year_hist single year 2023

bins = 12

f = ROOT.TFile(file)
t = f.Get("events")

f.Close()

h_2023 = year_hist("2023", UoB_year_files["2023"], UNIX_year_start, ROOT.kYellow-6, bins)



ymax = 1.1 * h_2023.GetMaximum()
h_2023.SetMaximum(ymax)
h_2023.SetMinimum(0)


c_2023 = ROOT.TCanvas("c_2023", "2023", 900, 600)

h_2023.Draw("HIST")

c_2023.Draw()

Warning in <TROOT::Append>: Replacing existing TH1: h_2023 (Potential memory leak).
Warning in <TCanvas::Constructor>: Deleting canvas with same name: c_2023


In [2]:
#year_hist single year 2023

bins = 12

f = ROOT.TFile(file)
t = f.Get("events")
f.Close()

h_2023 = year_hist("2023", UoB_year_files["2023"], UNIX_year_start, ROOT.kYellow-6, bins)

ymax = 1.1 * h_2023.GetMaximum()
h_2023.SetMaximum(ymax)
h_2023.SetMinimum(0)

c_2023 = ROOT.TCanvas("c_2023", "2023", 900, 600)

h_2023.Draw("HIST")


# -----------------------------
# Lightning data
# -----------------------------

lightning_2023 = extract_lightning(Bham_lightning_files["2023"])

year_seconds = seconds_in_year("2023")
month_seconds = year_seconds / 12


# -----------------------------
# Scaling
# -----------------------------

lightning_max = max(lightning_2023)
hist_max = h_2023.GetMaximum()

scale = hist_max / lightning_max


# -----------------------------
# Create scaled lightning graph
# -----------------------------

g_lightning = ROOT.TGraph(12)

for i, val in enumerate(lightning_2023):

    x = (i + 0.5) * month_seconds
    y = val * scale

    g_lightning.SetPoint(i, x, y)

g_lightning.SetMarkerStyle(20)
g_lightning.SetMarkerSize(1.2)
g_lightning.SetMarkerColor(ROOT.kRed)
g_lightning.SetLineColor(ROOT.kRed)


# -----------------------------
# Draw lightning
# -----------------------------

g_lightning.Draw("P SAME")


# -----------------------------
# Lightning axis
# -----------------------------

axis = ROOT.TGaxis(year_seconds, 0, year_seconds, hist_max, 0, lightning_max, 510, "+L")

axis.SetLineColor(ROOT.kRed)
axis.SetLabelColor(ROOT.kRed)
axis.SetTitle("Lightning hours per month")
axis.Draw()


# -----------------------------
# Legend
# -----------------------------

legend = ROOT.TLegend(0.7,0.75,0.88,0.88)

legend.AddEntry(h_2023,"Events","l")
legend.AddEntry(g_lightning,"Lightning","p")

legend.Draw()


# -----------------------------
# Correlation calculation
# -----------------------------

events = np.array([h_2023.GetBinContent(i+1) for i in range(12)])
lightning = lightning_2023

corr = np.corrcoef(events, lightning)[0,1]

print("Lightning vs Event correlation:", corr)


c_2023.Update()
c_2023.Draw()

Lightning vs Event correlation: 0.6740116736060382


In [56]:
file = np.loadtxt("Data/2021_bham_subset.csv", skiprows=1, usecols=range(4), delimiter = ',')
months = np.array(range(1,13))
lightning = file[:,3]
monthly_lightning = []
for i in range(12):
    monthly_lightning.append(int(np.sum(lightning[i::12])))

monthly_lightning

[0, 0, 19, 0, 502, 470, 291, 569, 68, 163, 37, 0]

In [77]:
def extract_lightning(data_file):
    file = np.loadtxt(data_file, skiprows=1, usecols=range(4), delimiter = ',')
    lightning = file[:,3]
    monthly_lightning = []
    for i in range(12):
        monthly_lightning.append(int(np.sum(lightning[i::12])))
    return np.array(monthly_lightning)

In [80]:
file = "Data/2015_bham_subset.csv"
extract_lightning(file)

array([ 154,    0,  353,   32,  253, 1092, 1197,  201,   81,    6,    0,
          0])

In [4]:
Bham_lightning_files = {
    "2015": "Data/2015_bham_subset.csv",
    "2016": "Data/2016_bham_subset.csv",
    "2017": "Data/2017_bham_subset.csv",
    "2018": "Data/2018_bham_subset.csv",
    "2019": "Data/2019_bham_subset.csv",
    "2020": "Data/2020_bham_subset.csv",
    "2021": "Data/2021_bham_subset.csv",
    "2022": "Data/2022_bham_subset.csv",
    "2023": "Data/2023_bham_subset.csv",
    "2024": "Data/2024_bham_subset.csv",
    "2025": "Data/2025_bham_subset.csv"
}

In [11]:
lightning_data = np.array([])
for year in UoB_year_files:
    if year == "2025":
        pass
    else:
        temp = extract_lightning(Bham_lightning_files[year])
        lightning_data = np.concatenate([lightning_data,temp])
lightning_data, len(lightning_data)

(array([1.540e+02, 0.000e+00, 3.530e+02, 3.200e+01, 2.530e+02, 1.092e+03,
        1.197e+03, 2.010e+02, 8.100e+01, 6.000e+00, 0.000e+00, 0.000e+00,
        0.000e+00, 0.000e+00, 0.000e+00, 7.200e+02, 8.570e+02, 4.480e+02,
        9.780e+02, 1.990e+02, 3.000e+00, 1.730e+02, 0.000e+00, 0.000e+00,
        0.000e+00, 0.000e+00, 0.000e+00, 0.000e+00, 2.290e+02, 8.110e+02,
        2.378e+03, 1.950e+02, 4.890e+02, 0.000e+00, 5.600e+01, 4.800e+01,
        0.000e+00, 0.000e+00, 1.700e+01, 4.500e+01, 2.840e+02, 0.000e+00,
        1.880e+02, 5.400e+01, 8.500e+01, 1.500e+02, 3.900e+01, 0.000e+00,
        2.700e+01, 0.000e+00, 3.000e+00, 4.000e+00, 1.500e+01, 5.890e+02,
        1.540e+03, 3.900e+02, 1.500e+02, 8.400e+01, 5.000e+00, 0.000e+00,
        0.000e+00, 7.600e+01, 0.000e+00, 6.600e+01, 2.700e+01, 7.200e+01,
        5.600e+01, 1.152e+03, 7.200e+01, 7.000e+00, 0.000e+00, 3.300e+01,
        0.000e+00, 3.000e+01, 0.000e+00, 0.000e+00, 3.870e+02, 6.810e+02,
        1.000e+00, 5.200e+01, 6.080e+0